In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_error
import warnings

warnings.filterwarnings("ignore")

# Load data
df = pd.read_csv(r'/content/clean_daily_sales.csv')

# --- FIX 1: RENAME COLUMNS ---
# Prophet MUST have 'ds' and 'y'. If your CSV names are different, change them here:
# Example: df = df.rename(columns={'Order Date': 'ds', 'Sales': 'y'})
# Let's try to force it if they aren't named right:
if 'ds' not in df.columns or 'y' not in df.columns:
    df.columns = ['ds', 'y'] # Assuming your CSV only has two columns in that order

# --- FIX 2: CONVERT DATE TO DATETIME ---
df['ds'] = pd.to_datetime(df['ds'])

# --- THE INTERPOLATION FIX ---
df['y'] = df['y'].replace(0, np.nan)
df['y'] = df['y'].interpolate(method='linear')
df['y'] = df['y'].fillna(method='bfill')

# 3. Fit Model
m = Prophet()
m.fit(df)

# 4. Forecast
future = m.make_future_dataframe(periods=90)
forecast = m.predict(future)

# 5. MAE
actuals = df['y']
predicted = forecast['yhat'][:len(df)]
mae = mean_absolute_error(actuals, predicted)

print(f"Your New MAE is: {mae}")

INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


Your New MAE is: 16951.4534785143


In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_error
import warnings

warnings.filterwarnings("ignore")

# 1. Load Data
df = pd.read_csv(r'/content/clean_daily_sales.csv')
df.columns = ['ds', 'y']
df['ds'] = pd.to_datetime(df['ds'])

# 2. Interpolation
df['y'] = df['y'].replace(0, np.nan)
df['y'] = df['y'].interpolate(method='linear')
df['y'] = df['y'].fillna(method='bfill')

# 3. Split Data
train = df[df['ds'] < '2018-06-01']
test = df[df['ds'] >= '2018-06-01']

# 4. Find Best Scale
best_mae = float('inf')
best_scale = 0.05
scales = [0.05, 0.1, 0.5]

for scale in scales:
    # Use 'm_temp' here
    m_temp = Prophet(changepoint_prior_scale=scale, yearly_seasonality=True)
    m_temp.add_country_holidays(country_name='IN') # Added holidays here too!
    m_temp.fit(train)

    future_test = test[['ds']]
    forecast_test = m_temp.predict(future_test)
    current_mae = mean_absolute_error(test['y'], forecast_test['yhat'])
    print(f"Scale: {scale} | MAE: {current_mae:,.2f}")

    if current_mae < best_mae:
        best_mae = current_mae
        best_scale = scale

print(f"\n>>> Best Scale Found: {best_scale} with MAE: {best_mae:,.2f}")

# 5. Train Final Model (Fixing the 'M' definition error)
df['cap'] = df['y'].max() * 1.5
df['floor'] = 0

m_final = Prophet(
    growth='logistic',
    changepoint_prior_scale=best_scale,
    yearly_seasonality=True
)

# FIXED: Use 'm_final' instead of 'm' and put it BEFORE .fit()
m_final.add_country_holidays(country_name='IN')
m_final.fit(df)

# 6. Predict 90 Days
future_90 = m_final.make_future_dataframe(periods=90)
future_90['cap'] = df['y'].max() * 1.5
future_90['floor'] = 0

forecast_90 = m_final.predict(future_90)
forecast_90['yhat'] = forecast_90['yhat'].clip(lower=0)

# 7. Final Results
future_only = forecast_90.tail(90)
wape = (np.sum(np.abs(test['y'] - forecast_test['yhat'])) / np.sum(test['y'])) * 100

# --- EXPORT WITH CONFIDENCE INTERVALS ---

# 1. Select the last 90 days with the range columns
# ds = Date, yhat = Prediction, yhat_lower = Min Range, yhat_upper = Max Range
future_results = forecast_90.tail(90)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

# 2. Rename columns for a professional Dashboard look
future_results.columns = ['Date', 'Predicted_Sales', 'Confidence_Lower', 'Confidence_Upper']

# 3. Final Safety Check: Ensure the lower range doesn't dip below 0
# (Since we used floor=0, this is mostly to clean up any tiny math noise)
future_results['Confidence_Lower'] = future_results['Confidence_Lower'].clip(lower=0)
future_results['Predicted_Sales'] = future_results['Predicted_Sales'].clip(lower=0)

# 4. Save to the final CSV
future_results.to_csv('predicted_sales_full_report.csv', index=False)

print("Your final report 'predicted_sales_full_report.csv' is ready for download!")

print(f"\n--- FINAL PREDICTION RESULTS ---")
print(f"Honest Test MAE: {best_mae:,.2f}")
print(f"Business Accuracy (WAPE): {wape:.2f}%")
print(f"Total Predicted Sales (Next 90 Days): {future_only['yhat'].sum():,.2f}")
print(f"Minimum Value in Forecast: {future_only['yhat'].min():,.2f}")

# --- THE EXPORT STEP ---
# 1. Take only the last 90 days (the future)
future_results = forecast_90.tail(90)[['ds', 'yhat']]

# 2. Rename them to look nice for your dashboard
future_results.columns = ['Date', 'Predicted_Sales']

# 3. Save to a new CSV file
future_results.to_csv('predicted_sales_final.csv', index=False)

print("Check the folder icon on the left! Your new file 'predicted_sales_final.csv' is ready.")


INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


Scale: 0.05 | MAE: 14,973.59


INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


Scale: 0.1 | MAE: 13,772.58


INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


Scale: 0.5 | MAE: 18,638.31

>>> Best Scale Found: 0.1 with MAE: 13,772.58
Your final report 'predicted_sales_full_report.csv' is ready for download!

--- FINAL PREDICTION RESULTS ---
Honest Test MAE: 13,772.58
Business Accuracy (WAPE): 18.29%
Total Predicted Sales (Next 90 Days): 2,030,183.88
Minimum Value in Forecast: 0.00
Check the folder icon on the left! Your new file 'predicted_sales_final.csv' is ready.


In [ ]:
# Calculate MAPE (Percentage Error)
mape = np.mean(np.abs((test['y'] - forecast_test['yhat']) / test['y'])) * 100
print(f"Your official MAPE is: {mape:.2f}%")


Your official MAPE is: 64.27%


In [ ]:
# WAPE = Sum of Errors / Sum of Actuals
wape = (np.sum(np.abs(test['y'] - forecast_test['yhat'])) / np.sum(test['y'])) * 100
print(f"Your Business Accuracy Error (WAPE): {wape:.2f}%")


Your Business Accuracy Error (WAPE): 17.79%
